In [2]:
# =============================================================================
# Project 2 (RAG), step 1: long-document corpus + chunking sweep with hybrid retrieval
# Run in Google Colab (T4). Paste each "# %% CELL" block into its own cell, in order.
# This notebook is self-contained: it does not need the Project 1 variables.
# =============================================================================

# %% CELL P2-1 -- install + imports
!pip install -q datasets sentence-transformers rank_bm25 pandas
import numpy as np

In [3]:
# %% CELL P2-2 -- pure helpers (no model, no dataset): offsets, chunking, gold labels, eval
import re, time, random

def assemble_doc(paras):
    """Join paragraphs into one long document and return the char offset where each starts."""
    offsets, pos = [], 0
    for p in paras:
        offsets.append(pos)
        pos += len(p) + 2            # +2 for the "\n\n" separator
    return "\n\n".join(paras), offsets

WORD_RE = re.compile(r"\S+")

def chunk_words_with_spans(text, chunk_size=100, overlap=20):
    """Sliding window over words. Returns [(start_char, end_char), ...] per chunk.
    Stops as soon as a window reaches the last word, so there is no redundant tail chunk."""
    assert 0 <= overlap < chunk_size
    spans = [m.span() for m in WORD_RE.finditer(text)]
    step = chunk_size - overlap
    chunks, i = [], 0
    while i < len(spans):
        j = min(i + chunk_size, len(spans))
        chunks.append((spans[i][0], spans[j - 1][1]))
        if j == len(spans):
            break
        i += step
    return chunks

def build_chunks(docs, chunk_size, overlap):
    chunks = []
    for doc_id, text in docs.items():
        for k, (s, e) in enumerate(chunk_words_with_spans(text, chunk_size, overlap)):
            chunks.append({"id": f"{doc_id}#{k}", "doc_id": doc_id,
                           "start": s, "end": e, "text": text[s:e]})
    return chunks

def attach_gold(questions, chunks):
    """Gold = every chunk that FULLY contains the answer span. If none does (the answer was
    split across a boundary), gold is empty and the question counts as a miss."""
    by_doc = {}
    for c in chunks:
        by_doc.setdefault(c["doc_id"], []).append(c)
    out = []
    for q in questions:
        gold = {c["id"] for c in by_doc[q["doc_id"]]
                if c["start"] <= q["start"] and q["end"] <= c["end"]}
        out.append({**q, "gold": gold})
    return out

def evaluate(retrieve_fn, questions, k_values=(1, 5, 10)):
    hits = {k: [] for k in k_values}
    rr, lat = [], []
    for q in questions:
        t0 = time.perf_counter()
        ranked = retrieve_fn(q["question"], top_k=max(k_values))
        lat.append(time.perf_counter() - t0)
        gold = q["gold"]
        for k in k_values:
            hits[k].append(1 if any(c in gold for c in ranked[:k]) else 0)
        rank = next((i + 1 for i, c in enumerate(ranked) if c in gold), None)
        rr.append(1.0 / rank if rank else 0.0)
    lat.sort()
    out = {f"recall@{k}": sum(v) / len(v) for k, v in hits.items()}
    out["mrr"] = sum(rr) / len(rr)
    out["p50_ms"] = lat[len(lat) // 2] * 1000
    out["p95_ms"] = lat[int(len(lat) * 0.95)] * 1000
    return out

# --- retrieval helpers carried over from Project 1 ---
STOPWORDS = {"the","a","an","of","in","on","at","to","is","was","were","are","and","or",
             "for","by","with","as","what","which","who","when","where","how","did","does",
             "do","from","that","this","it","its","be","been","has","have","had"}

def better_tokenize(text):
    return [t for t in re.findall(r"\w+", text.lower()) if t not in STOPWORDS]

def rrf_fuse(ranked_lists, weights, k_constant=60, top_k=10):
    scores = {}
    for ranked, w in zip(ranked_lists, weights):
        for rank, pid in enumerate(ranked):
            scores[pid] = scores.get(pid, 0.0) + w / (k_constant + rank + 1)
    return sorted(scores, key=lambda pid: -scores[pid])[:top_k]

In [4]:
# %% CELL P2-3 -- load SQuAD v2 articles as LONG documents, with answer offsets
from datasets import load_dataset

def build_corpus(n_articles=150, n_questions=500, seed=42):
    ds = load_dataset("rajpurkar/squad_v2", split="train")
    by_title = {}
    for row in ds:
        by_title.setdefault(row["title"], []).append(row)

    rng = random.Random(seed)
    titles = rng.sample(sorted(by_title), n_articles)

    docs, questions, skipped = {}, [], 0
    for t_i, title in enumerate(titles):
        rows = by_title[title]
        paras, pidx = [], {}
        for r in rows:
            if r["context"] not in pidx:
                pidx[r["context"]] = len(paras)
                paras.append(r["context"])
        text, offsets = assemble_doc(paras)
        doc_id = f"d{t_i}"
        docs[doc_id] = text

        for r in rows:
            if not r["answers"]["text"]:          # unanswerable: kept for the abstain-gate step later
                continue
            a_text = r["answers"]["text"][0]
            start = offsets[pidx[r["context"]]] + r["answers"]["answer_start"][0]
            end = start + len(a_text)
            if text[start:end] != a_text:         # offset sanity check; skip anything inconsistent
                skipped += 1
                continue
            questions.append({"question": r["question"], "doc_id": doc_id,
                              "start": start, "end": end,
                              "answers": list(r["answers"]["text"])})

    questions = rng.sample(questions, min(n_questions, len(questions)))
    words = sum(len(t.split()) for t in docs.values())
    print(f"{len(docs)} documents, avg {words // len(docs)} words each; "
          f"{len(questions)} questions (skipped {skipped} with bad offsets)")
    return docs, questions

docs, questions = build_corpus()

README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

squad_v2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.4MB            

squad_v2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

squad_v2/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 1.35MB            

squad_v2/validation-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

150 documents, avg 5126 words each; 500 questions (skipped 0 with bad offsets)


In [5]:
# %% CELL P2-4 -- chunking sweep: chunk size x overlap, dense vs hybrid
import pandas as pd
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

model = SentenceTransformer("BAAI/bge-small-en-v1.5")   # max 512 tokens: keep chunks <= ~300 words

def build_retrievers(chunks):
    ids = [c["id"] for c in chunks]
    texts = [c["text"] for c in chunks]
    emb = model.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    bm25 = BM25Okapi([better_tokenize(t) for t in texts])

    def dense(q, top_k=10):
        qe = model.encode([q], normalize_embeddings=True)[0]
        return [ids[i] for i in np.argsort(-(emb @ qe))[:top_k]]

    def sparse(q, top_k=10):
        s = bm25.get_scores(better_tokenize(q))
        return [ids[i] for i in np.argsort(-s)[:top_k]]

    def hybrid(q, top_k=10):
        return rrf_fuse([dense(q, 50), sparse(q, 50)], [1.0, 1.0], top_k=top_k)

    return dense, hybrid

rows = []
for size in (50, 100, 200, 300):
    for frac in (0.0, 0.2):
        overlap = int(size * frac)
        chunks = build_chunks(docs, size, overlap)
        qs = attach_gold(questions, chunks)
        split_pct = 100 * sum(1 for q in qs if not q["gold"]) / len(qs)
        dense, hybrid = build_retrievers(chunks)
        for name, fn in (("dense", dense), ("hybrid", hybrid)):
            res = evaluate(fn, qs)
            rows.append({"chunk_words": size, "overlap": overlap, "n_chunks": len(chunks),
                         "answer_split_%": round(split_pct, 1), "retriever": name, **res})
            print(f"size={size:3d} overlap={overlap:3d} {name:6s} "
                  f"r@5={res['recall@5']:.3f} split={split_pct:.1f}%")

df = pd.DataFrame(rows).round(3)
print(df.to_string(index=False))
df.to_csv("project2_chunking_sweep.csv", index=False)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

KeyboardInterrupt: 

In [6]:
# %% CELL P2-5 -- install (active line, run it) + config + check step-1 state
!pip install -q sentence-transformers rank_bm25 pandas

CHUNK_SIZE, OVERLAP = 100, 20     # my pick: the step-1 sweep never finished. Change after you see its table.
POOLS = (10, 20, 50)              # how many hybrid candidates the re-ranker sees
RERANKERS = {                     # name -> model id
    "minilm-l6":         "cross-encoder/ms-marco-MiniLM-L-6-v2",   # small, fast
    "bge-reranker-base": "BAAI/bge-reranker-base",                 # larger, usually stronger
}

needed = ["docs", "questions", "build_chunks", "attach_gold", "evaluate", "rrf_fuse", "better_tokenize"]
missing = [n for n in needed if n not in globals()]
if missing:
    raise RuntimeError(f"Missing {missing}. Run step-1 cells P2-1, P2-2, P2-3 first (same session).")
print("step-1 state found:", len(docs), "documents,", len(questions), "questions")


step-1 state found: 150 documents, 500 questions


In [7]:
# %% CELL P2-6 -- re-rank logic (pure functions: no model, no dataset)
def rerank_ids(question, candidate_ids, id2text, score_fn, top_k=10):
    """Score every (question, chunk) pair with score_fn, return candidate ids best-first."""
    if not candidate_ids:
        return []
    pairs = [(question, id2text[c]) for c in candidate_ids]
    scores = score_fn(pairs)
    order = sorted(range(len(candidate_ids)), key=lambda i: -scores[i])
    return [candidate_ids[i] for i in order[:top_k]]

def make_reranked(candidate_fn, pool, id2text, score_fn):
    """retrieve_fn(question, top_k) -> ids. Takes `pool` candidates, re-ranks, returns top_k."""
    def retrieve(question, top_k=10):
        candidates = candidate_fn(question, top_k=pool)
        return rerank_ids(question, candidates, id2text, score_fn, top_k=top_k)
    return retrieve

In [8]:
# %% CELL P2-7 -- build the hybrid retriever, then run baseline vs re-rankers
import time
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

chunks = build_chunks(docs, CHUNK_SIZE, OVERLAP)
qs = attach_gold(questions, chunks)
ids = [c["id"] for c in chunks]
texts = [c["text"] for c in chunks]
id2text = dict(zip(ids, texts))
miss = 100 * sum(1 for q in qs if not q["gold"]) / len(qs)
print(f"{len(chunks)} chunks (size={CHUNK_SIZE}, overlap={OVERLAP}); {miss:.1f}% of questions have no single gold chunk")

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
emb = embedder.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
bm25 = BM25Okapi([better_tokenize(t) for t in texts])

def dense(q, top_k=10):
    qe = embedder.encode([q], normalize_embeddings=True)[0]
    return [ids[i] for i in np.argsort(-(emb @ qe))[:top_k]]

def sparse(q, top_k=10):
    return [ids[i] for i in np.argsort(-bm25.get_scores(better_tokenize(q)))[:top_k]]

def hybrid(q, top_k=10):
    return rrf_fuse([dense(q, 50), sparse(q, 50)], [1.0, 1.0], top_k=top_k)

KS = (1, 3, 5, 10)
rows = []

base = evaluate(hybrid, qs, k_values=KS)
rows.append({"config": "hybrid (no re-rank)", "pool": "-", **base})

# Ceiling: a re-ranker can only reorder what the pool contains, so recall@pool is the best it can do.
print("\nCeiling (hybrid recall@pool):")
for pool in POOLS:
    ceil = evaluate(hybrid, qs, k_values=(pool,))[f"recall@{pool}"]
    print(f"  recall@{pool} = {ceil:.3f}")

for name, model_id in RERANKERS.items():
    ce = CrossEncoder(model_id, max_length=512)
    score_fn = lambda pairs, ce=ce: ce.predict(pairs, batch_size=32, show_progress_bar=False)
    for q in qs[:3]:                                   # warm-up so GPU start-up is not in the timings
        score_fn([(q["question"], texts[0])] * 8)
    for pool in POOLS:
        fn = make_reranked(hybrid, pool, id2text, score_fn)
        res = evaluate(fn, qs, k_values=KS)
        rows.append({"config": f"hybrid + {name}", "pool": pool, **res})
        print(f"{name:18s} pool={pool:2d}  r@1={res['recall@1']:.3f}  r@5={res['recall@5']:.3f}  "
              f"mrr={res['mrr']:.3f}  p50={res['p50_ms']:.0f}ms")

df = pd.DataFrame(rows).round(3)
print("\n", df.to_string(index=False))
df.to_csv("project2_reranker.csv", index=False)

9652 chunks (size=100, overlap=20); 0.0% of questions have no single gold chunk


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Ceiling (hybrid recall@pool):
  recall@10 = 0.916
  recall@20 = 0.956
  recall@50 = 0.982


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

bge-reranker-base  pool=10  r@1=0.488  r@5=0.820  mrr=0.629  p50=149ms
bge-reranker-base  pool=20  r@1=0.416  r@5=0.750  mrr=0.550  p50=263ms
bge-reranker-base  pool=50  r@1=0.340  r@5=0.600  mrr=0.455  p50=613ms

                     config pool  recall@1  recall@3  recall@5  recall@10   mrr  p50_ms  p95_ms
       hybrid (no re-rank)    -     0.616     0.788     0.844      0.916 0.717  39.539  58.671
        hybrid + minilm-l6   10     0.770     0.880     0.902      0.916 0.828  77.898 100.514
        hybrid + minilm-l6   20     0.782     0.894     0.924      0.948 0.844  98.985 126.388
        hybrid + minilm-l6   50     0.790     0.902     0.932      0.958 0.853 140.772 174.126
hybrid + bge-reranker-base   10     0.488     0.728     0.820      0.916 0.629 149.176 178.527
hybrid + bge-reranker-base   20     0.416     0.614     0.750      0.884 0.550 263.086 315.239
hybrid + bge-reranker-base   50     0.340     0.504     0.600      0.748 0.455 613.186 691.500


In [9]:
from google.colab import files
files.download("project2_reranker.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>